### debugging

In [ ]:
import pandas as pd
from ast import literal_eval

CSV_PATH = 'data/emplo.csv'
USE_COLS = [
    'employee_Id',       # your ID column
    'sector', 'contract_type', 'edu_value', 'city',
    'years_experience', 'salary', 'technical_skills'
]
HIERARCHY = ['sector', 'contract_type', 'edu_value', 'city', 'years_experience']

def preprocess():
    df = pd.read_csv(CSV_PATH, usecols=USE_COLS)
    df['sector'] = df['sector'].fillna('Unknown').astype(str)
    
    def split_skills(x):
        if pd.isna(x):
            return ['Unknown']
        try:
            lst = literal_eval(x)
            if isinstance(lst, (list, tuple)):
                return [str(s).strip() for s in lst]
        except Exception:
            pass
        return [s.strip() for s in str(x).split(',') if s.strip()]
    
    # explode skills into one-per-row
    df = df.assign(
        technical_skills=df['technical_skills'].apply(split_skills)
    ).explode('technical_skills')
    return df

def build_transition_model(df, level=0):
    if level >= len(HIERARCHY):
        return {}
    
    attr = HIERARCHY[level]
    nodes = {}
    
    for val, grp in df.groupby(attr, dropna=False):
        key = str(val) if not pd.isna(val) else 'Unknown'
        
        # expected values for everything not yet grouped
        remaining = [
            a for a in HIERARCHY + ['salary', 'technical_skills']
            if a not in HIERARCHY[:level+1]
        ]
        expected = {}
        for r in remaining:
            if r in ['years_experience', 'salary', 'edu_value']:
                expected[r] = round(grp[r].mean(), 1)
            else:
                counts = grp[r].fillna('Unknown').astype(str).value_counts()
                total  = counts.sum()
                expected[r] = {k: round(v/total, 2) for k, v in counts.items()}
        
        # at last hierarchy level, collect unique employee IDs
        extras = {}
        if level == len(HIERARCHY) - 1:
            unique_ids = sorted(set(grp['employee_Id']))
            extras['employee_ids'] = unique_ids
        
        node = {
            'expected_values': expected,
            **extras
        }
        # recurse
        node['clusters'] = build_transition_model(grp, level+1)
        nodes[key] = node
    
    return nodes

def print_model(model, depth=0, max_depth=2, indent=0):
    if depth > max_depth:
        return
    for k, v in model.items():
        print(' '*indent + f"Cluster: {k}")
        for a, val in v['expected_values'].items():
            if isinstance(val, dict):
                items = ', '.join(f"{sk}: {pct:.2f}" for sk, pct in val.items())
                print(' '*(indent+2) + f"{a}: {{{items}}}")
            else:
                print(' '*(indent+2) + f"{a}: {val:.1f}")
        if 'employee_ids' in v:
            print(' '*(indent+2) + f"employee_ids: {v['employee_ids']}")
        print()
        print_model(v['clusters'], depth+1, max_depth, indent+4)
# Export the transition model to a JSON file
def export_transition_model(transition_model, file_path='transition_model.json'):
    """
    Exports the transition model as a JSON file.

    Args:
        transition_model (dict): The transition model to export.
        file_path (str): The file path to save the JSON file.
    """
    with open(file_path, 'w') as json_file:
        json.dump(transition_model, json_file, indent=4)
    print(f"Transition model exported to {file_path}")


if __name__ == '__main__':
    df = preprocess()
    model = build_transition_model(df)
    print_model(model, max_depth=2)
    export_transition_model(model, file_path='transition_model2.json')

    


In [ ]:
import json

# Export the transition model to a JSON file
def export_transition_model(transition_model, file_path='transition_model.json'):
    """
    Exports the transition model as a JSON file.

    Args:
        transition_model (dict): The transition model to export.
        file_path (str): The file path to save the JSON file.
    """
    with open(file_path, 'w') as json_file:
        json.dump(transition_model, json_file, indent=4)
    print(f"Transition model exported to {file_path}")

# Import the transition model from a JSON file


# Example usage:
# Export the transition model
export_transition_model(transition_model, file_path='transition_model.json')

# Import the transition model in another notebook
# (Run this in the other notebook)
# imported_model = import_transition_model(file_path='transition_model.json')

In [ ]:

# use this function in the other files to import the json file

def import_transition_model(file_path='transition_model.json'):
    """
    Imports the transition model from a JSON file.

    Args:
        file_path (str): The file path of the JSON file to import.

    Returns:
        dict: The imported transition model.
    """
    with open(file_path, 'r') as json_file:
        transition_model = json.load(json_file)
    print(f"Transition model imported from {file_path}")
    return transition_model

In [19]:
import pandas as pd
import json
from ast import literal_eval

CSV_PATH = 'data/jobs.csv'
USE_COLS = [
    'job_Id',             # your job ID column
    'sector', 
    'type of contract', 
    'edu_value', 
    'job location', 
    'experience_min_req', 
    'experience_max_req',
    'salary', 
    'technical_skills'
]
# Mirror the same hierarchy
HIERARCHY = ['sector', 'contract_type', 'edu_value', 'city', 'years_experience']

def preprocess_jobs():
    df = pd.read_csv(CSV_PATH, usecols=USE_COLS)
    # rename to match hierarchy names
    df = df.rename(columns={
        'type of contract': 'contract_type',
        'job location':       'city',
        'experience_min_req': 'exp_min',
        'experience_max_req': 'exp_max',
    })
    # compute a single years_experience value
    df['years_experience'] = ((df['exp_min'].fillna(0) + df['exp_max'].fillna(0)) / 2).round(1)
    # clean up strings
    df['sector']        = df['sector'].fillna('Unknown').astype(str)
    df['city']          = df['city'].fillna('Unknown').astype(str)
    # split & explode skills
    def split_skills(x):
        if pd.isna(x):
            return ['Unknown']
        try:
            lst = literal_eval(x)
            if isinstance(lst, (list, tuple)):
                return [s.strip() for s in lst]
        except Exception:
            pass
        return [s.strip() for s in str(x).split(',') if s.strip()]

    df = df.assign(
        technical_skills=df['technical_skills'].apply(split_skills)
    ).explode('technical_skills')
    return df[[
        'job_Id', 'sector', 'contract_type', 'edu_value', 
        'city', 'years_experience', 'salary', 'technical_skills'
    ]]

def build_job_transition_model(df, level=0):
    if level >= len(HIERARCHY):
        return {}
    
    attr = HIERARCHY[level]
    nodes = {}
    
    for val, grp in df.groupby(attr, dropna=False):
        key = str(val) if pd.notna(val) else 'Unknown'
        
        # compute expected values for remaining attrs
        remaining = [
            a for a in HIERARCHY + ['salary', 'technical_skills']
            if a not in HIERARCHY[:level+1]
        ]
        expected = {}
        for r in remaining:
            if r in ['years_experience', 'salary', 'edu_value']:
                expected[r] = round(grp[r].mean(), 1)
            else:
                counts = grp[r].fillna('Unknown').astype(str).value_counts()
                total  = counts.sum()
                expected[r] = {k: round(v/total, 2) for k, v in counts.items()}
        
        # at leaf level, collect job IDs (deduped)
        extras = {}
        if level == len(HIERARCHY) - 1:
            extras['job_ids'] = sorted(set(grp['job_Id']))
        
        node = {
            'expected_values': expected,
            **extras
        }
        node['clusters'] = build_job_transition_model(grp, level+1)
        nodes[key] = node
    
    return nodes

def print_model(model, depth=0, max_depth=2, indent=0):
    if depth > max_depth:
        return
    for k, v in model.items():
        print(' ' * indent + f"Cluster: {k}")
        for a, val in v['expected_values'].items():
            if isinstance(val, dict):
                items = ', '.join(f"{x}: {pct:.2f}" for x, pct in val.items())
                print(' ' * (indent+2) + f"{a}: {{{items}}}")
            else:
                print(' ' * (indent+2) + f"{a}: {val:.1f}")
        if 'job_ids' in v:
            print(' ' * (indent+2) + f"job_ids: {v['job_ids']}")
        print()
        print_model(v['clusters'], depth+1, max_depth, indent+4)

def export_transition_model(transition_model, file_path='job_transition_model.json'):
    with open(file_path, 'w') as f:
        json.dump(transition_model, f, indent=2)
    print(f"Exported job transition model to {file_path}")

if __name__ == '__main__':
    df_jobs = preprocess_jobs()
    job_model = build_job_transition_model(df_jobs)
    
    # print a truncated view
    print_model(job_model, max_depth=2)
    # export full JSON
    export_transition_model(job_model)


Cluster: Aerospace & Defense
  contract_type: {Freelance: 0.22, Stage: 0.21, CDD: 0.20, Alternance: 0.19, CDI: 0.17}
  edu_value: 16.5
  city: {Algiers: 0.04, Annaba: 0.04, Saïda: 0.04, Sétif: 0.04, Tiaret: 0.04, Béjaïa: 0.04, In Salah: 0.04, Constantine: 0.04, Tizi Ouzou: 0.04, Timimoun: 0.03, Mostaganem: 0.03, Adrar: 0.03, Blida: 0.03, Aïn Témouchent: 0.03, Jijel: 0.03, Chlef: 0.03, Oran: 0.03, Biskra: 0.03, Laghouat: 0.03, Ghardaïa: 0.03, Naama: 0.03, El Oued: 0.03, Tlemcen: 0.03, Tamanrasset: 0.03, Djanet: 0.02, Tébessa: 0.02, Skikda: 0.02, Ouargla: 0.02, M'sila: 0.02, Batna: 0.02, El Tarf: 0.02, Illizi: 0.02, Tindouf: 0.02, Béchar: 0.01}
  years_experience: 5.9
  salary: 48682.4
  technical_skills: {French Bilingual: 0.20, Team Leadership: 0.20, Microsoft Office: 0.20, Report Writing: 0.20, Project Management: 0.20}

    Cluster: Alternance
      edu_value: 16.6
      city: {Algiers: 0.06, Annaba: 0.06, Sétif: 0.06, Aïn Témouchent: 0.05, Saïda: 0.05, Tiaret: 0.05, Tébessa: 0.05, O